### 20th February 2025

Plot alignments of the kmers used as the referneces to the sequences used as the bait or kmer.

Her we aligned the kmers to both the anchor sequence Mattis, as well as the ref lacking the deletion present in mattis so we could understand how the kmer distribution would look like whn we are dealing with a deletion.

In [ ]:
## define path to files
import pandas as pd
import glob
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
spadesOutDir="/Volumes/Group-Scratch/Cristobal-Uauy/waweru/2023_11_chr2D_CS_635_651/kmers_from_fasta/chinesespring_chr4B_30860000-30910000/spades_contigs_assembly/ibspy_files_analysis"
blastfile="chinesespringchr4B_30860000-30910000_ContigkmersSeqsToAnchorSeq_blast.out"
inputFile=f"{spadesOutDir}/{blastfile}"


In [ ]:
# import data into dataframe

# Load BLAST format 6 into a DataFrame
columns = [
    "query_id", "subject_id", "percent_identity", "alignment_length", 
    "mismatches", "gap_opens", "query_start", "query_end", 
    "subject_start", "subject_end", "e_value", "bit_score"
]
blast_df = pd.read_csv(inputFile, sep="\t", names=columns)




In [ ]:
blast_df.head()

filter and prepare data

In [ ]:
# Example: Filter for high-quality alignments
filtered_df = blast_df[blast_df["e_value"] < 1e-5]

# Prepare additional metrics
filtered_df["alignment_midpoint"] = (filtered_df["query_start"] + filtered_df["query_end"]) / 2

# Calculate midpoint for each alignment
filtered_df["midpoint"] = (filtered_df["subject_start"] + filtered_df["subject_end"]) / 2


# Correct reverse alignments
filtered_df["adjusted_start"] = filtered_df[["subject_start", "subject_end"]].min(axis=1)
filtered_df["adjusted_end"] = filtered_df[["subject_start", "subject_end"]].max(axis=1)


In [ ]:
#save to csv file


filtered_df.to_csv(f"{spadesOutDir}/chinesespringchr4B_30860000-30910000_ContigkmersSeqsToAnchorSeq_blast.csv")


A segment plot visualizes each alignment's range on the reference sequence

In [ ]:
import matplotlib.pyplot as plt

# Plot alignment ranges on the reference
plt.figure(figsize=(40, 2))
for index, row in filtered_df.iterrows():
    plt.plot(
        [row["subject_start"], row["subject_end"]],  # Start and end positions
        [row["subject_id"], row["subject_id"]],      # Subject (reference ID)
        marker="o", color="blue", alpha=0.7
    )

plt.title("Alignment Locations on Reference Sequences")
plt.xlabel("Reference Position")
plt.ylabel("Reference ID")
plt.grid(axis="x")
plt.show()


If you want to show the cumulative coverage of alignments on the reference sequence:

Aggregate alignment counts across reference positions.
Plot coverage along the reference sequence

In [ ]:
import numpy as np

# Create a coverage dictionary for each subject
coverage_dict = {}
for subject_id in filtered_df["subject_id"].unique():
    subject_df = filtered_df[filtered_df["subject_id"] == subject_id]
    coverage = np.zeros(subject_df["subject_end"].max() + 1)
    for _, row in subject_df.iterrows():
        coverage[row["subject_start"]:row["subject_end"]] += 1
    coverage_dict[subject_id] = coverage

# Plot coverage for each reference
plt.figure(figsize=(60, 8))
for subject_id, coverage in coverage_dict.items():
    plt.plot(coverage, label=f"{subject_id}")

plt.title("Coverage Across Reference Sequences")
plt.xlabel("Position on Reference")
plt.ylabel("Coverage")
plt.legend()
plt.show()


A heatmap can visualize the density of alignments across the reference

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Assuming filtered_df is your BLAST format 6 dataframe
# Columns: subject_id, subject_start, subject_end

# Compute the maximum length for the reference
max_length = filtered_df["adjusted_end"].max()

# Initialize coverage dictionary
coverage_dict = {}

# Populate coverage for each reference
for subject_id in filtered_df["subject_id"].unique():
    # Create an array for the length of the reference
    coverage = np.zeros(max_length)
    
    # Extract alignments for this subject
    subject_alignments = filtered_df[filtered_df["subject_id"] == subject_id]
    
    for _, row in subject_alignments.iterrows():
        # Mark coverage for the alignment
        coverage[row["adjusted_start"] - 1 : row["adjusted_end"]] += 1  # Adjust 0-based indexing
    
    # Store the coverage
    coverage_dict[subject_id] = coverage

# Create the heatmap matrix
heatmap_matrix = np.array(list(coverage_dict.values()))




In [ ]:
# Plot heatmap
plt.figure(figsize=(80, 4))
sns.heatmap(heatmap_matrix, cmap="viridis", xticklabels=50, yticklabels=coverage_dict.keys())
plt.title("Alignment Coverage Heatmap Across Reference Anchor Sequence")
plt.xlabel("Position on Anchor Sequence")
plt.ylabel("Reference ID")
plt.show()

Visualizing the alingmnets as a scatter plot

In [ ]:
# Calculate midpoint for each alignment
filtered_df["midpoint"] = (filtered_df["subject_start"] + filtered_df["subject_end"]) / 2

# Plot the scatter plot
plt.figure(figsize=(60, 2))

# Scatter plot with percentage identity as color
scatter = plt.scatter(
    filtered_df["midpoint"],
    filtered_df["subject_id"],
    c=filtered_df["percent_identity"],
    cmap="viridis",
    s=50,  # Size of the dots
    edgecolor="k",
    alpha=0.7
)

# Add a color bar to indicate percentage identity
cbar = plt.colorbar(scatter)
cbar.set_label("Percentage Identity (%)")

# Add labels and title
plt.xlabel("Position on Anchor Sequence (Midpoint)")
plt.ylabel("Anchor Sequence ID")
plt.title("BLAST Alignment Scatter Plot")
plt.grid(alpha=0.3)

In [ ]:
filtered_df.to_csv(f"{spadesOutDir}/CS_blast_out.csv")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pandas as pd

df = filtered_df

# Initialize the plot
#fig, ax = plt.subplots(figsize=(50, 50))
# Initialize the plot with specific dimensions
fig, ax = plt.subplots(figsize=(70, 15))  # Width=60 inches, Height=50 inches


# Create a colormap
norm = mcolors.Normalize(vmin=df["percent_identity"].min(), vmax=df["percent_identity"].max())
cmap = plt.cm.viridis

# Plot each alignment as a line
for _, row in df.iterrows():
    ax.plot(
        [row["subject_start"], row["subject_end"]],  # x-coordinates
        [row["subject_start"], row["subject_start"]],  # y-coordinates (fixed for horizontal lines)
        color=cmap(norm(row["percent_identity"])),  # Color based on percentage identity
        linewidth=2,
    )

# Add color bar explicitly referencing the scalar mappable
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])  # Empty array since data is already in the plot
cbar = fig.colorbar(sm, ax=ax)  # Associate the colorbar with the current Axes
cbar.set_label("Percentage Identity (%)")

# Set ticks every 50
tick_interval_x = 50
tick_interval_y = 200
max_position = max(df["subject_end"].max(), df["subject_start"].max()) + 50
ax.set_xticks(range(0, int(max_position) + tick_interval_x, tick_interval_x))
ax.set_yticks(range(0, int(max_position) + tick_interval_y, tick_interval_y))

# Rotate x-axis labels
plt.xticks(rotation=90, fontsize=5, ha='right')

# Add labels and grid
ax.set_xlabel("Alignment End Position")
ax.set_ylabel("Alignment Start Position")
ax.set_title("BLAST Alignment Coverage on Anchor Sequence")
ax.grid(alpha=0.3)

# Set limits and aspect ratio
ax.set_xlim(0, max_position)
ax.set_ylim(0, max_position)
plt.tight_layout()

# Save the plot to a PDF
output_file = f"{spades_dir}/CS_alignment_coverage_plot.pdf"
plt.savefig(output_file, format="pdf", dpi=300) 

# Optionally, close the plot to free resources
plt.close(fig)

print(f"Plot saved to {output_file}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pandas as pd

# Example BLAST output dataframe
data = {
    "subject_id": ["ref1"] * 4,  # Example assumes a single reference
    "subject_start": [100, 300, 50, 600],
    "subject_end": [200, 400, 150, 800],
    "pident": [98.5, 87.2, 92.3, 88.5]
}
df = pd.DataFrame(data)

# Initialize the plot
fig, ax = plt.subplots(figsize=(10, 8))

# Create a colormap
norm = mcolors.Normalize(vmin=df["pident"].min(), vmax=df["pident"].max())
cmap = plt.cm.viridis

# Plot each alignment as a line
for _, row in df.iterrows():
    ax.plot(
        [row["subject_start"], row["subject_end"]],  # x-coordinates
        [row["subject_start"], row["subject_start"]],  # y-coordinates (fixed for horizontal lines)
        color=cmap(norm(row["pident"])),  # Color based on percentage identity
        linewidth=2,
    )

# Add color bar explicitly referencing the scalar mappable
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])  # Empty array since data is already in the plot
cbar = fig.colorbar(sm, ax=ax)  # Associate the colorbar with the current Axes
cbar.set_label("Percentage Identity (%)")

# Add labels and grid
ax.set_xlabel("Alignment End Position")
ax.set_ylabel("Alignment Start Position")
ax.set_title("BLAST Alignment Coverage on Reference")
ax.grid(alpha=0.3)

# Set limits and aspect ratio
ax.set_xlim(0, df["subject_end"].max() + 50)
ax.set_ylim(0, df["subject_start"].max() + 50)
plt.tight_layout()

# Show the plot
plt.show()

